In [1]:
# from torch_geometric.datasets import TUDataset
import torch
import sys
sys.path.append('..')
from dhg import Hypergraph
from hypgs.models.hsn_pyg import HSN
from hypgs.utils.data import HGDataset
from torch_geometric.loader import DataLoader
from torch_geometric.data import Dataset
import pytorch_lightning as pl
from pytorch_lightning.callbacks import EarlyStopping
from torch.utils.data import random_split
from pytorch_lightning.loggers import TensorBoardLogger
from dhg.data import Cooking200, CoauthorshipCora
# from torch_geometric.data import ClusterData, ClusterLoader
from hypgs.utils.data import HGDatasetFromDGL
# from torch_geometric.loader import NeighborSampler
from torch_geometric.loader import NeighborLoader


In [2]:
data = Cooking200()
X, lbl = torch.eye(data["num_vertices"]), data["labels"]
HG = Hypergraph(data["num_vertices"], data["edge_list"])
Y = torch.zeros(HG.num_e, X.shape[1])
train_mask = data["train_mask"]
val_mask = data["val_mask"]
test_mask = data["test_mask"]

In [3]:
print(f"{train_mask.sum()=}, {val_mask.sum()=}, {test_mask.sum()=}")

train_mask.sum()=tensor(200), val_mask.sum()=tensor(200), test_mask.sum()=tensor(7003)


In [4]:
dataset = HGDatasetFromDGL(HG, X, Y, lbl)

In [5]:
# num_nodes = dataset[0].num_nodes
# num_train = int(num_nodes * 0.8)
# num_val = int(num_nodes * 0.1)

# perm = torch.randperm(num_nodes)
# train_idx = perm[:num_train]
# val_idx = perm[num_train:num_train + num_val]
# test_idx = perm[num_train + num_val:]
train_idx = torch.where(train_mask)[0]
val_idx = torch.where(val_mask)[0]
test_idx = torch.where(test_mask)[0]

In [6]:
train_loader = NeighborLoader(
    dataset[0],
    input_nodes=train_idx,
    num_neighbors=[10, 10],
    batch_size=512,
    shuffle=True,
    num_workers=4
)

val_loader = NeighborLoader(
    dataset[0],
    input_nodes=val_idx,
    num_neighbors=[10, 10],
    batch_size=512,
    shuffle=False,
    num_workers=4
)

test_loader = NeighborLoader(
    dataset[0],
    input_nodes=test_idx,
    num_neighbors=[10, 10],
    batch_size=512,
    shuffle=False,
    num_workers=4
)


/gpfs/gibbs/pi/krishnaswamy_smita/xingzhi/.conda_envs/pyg/lib/python3.9/site-packages/torch/utils/data/dataloader.py:557: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 1, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(_create_warning_msg(


In [7]:
lbl.unique()

tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
        18, 19])

In [8]:
batch = next(iter(train_loader))

In [9]:
model = HSN(in_channels=7403, 
              hidden_channels=16,
              out_channels = 20, 
              trainable_laziness = False,
              trainable_scales = False, 
              activation = 'modulus', 
              fixed_weights=True, 
              layout=['hsm'], 
              normalize='right', 
              pooling=None,
              task='classification',
        )

In [10]:
# Early stopping callback based on validation loss
early_stopping_callback = EarlyStopping(monitor='val_loss', patience=10, verbose=True, mode='min')
logger = TensorBoardLogger("lightning_logs", name="my_model_cooking")

trainer = pl.Trainer(
    max_epochs=100,  # Adjust as necessary
    logger=logger,
    callbacks=[early_stopping_callback]
)

# Assuming you have defined train_loader and val_loader
trainer.fit(model, train_loader, val_loader)  # Corrected argument names
# Test the model
trainer.test(model, test_loader)

/gpfs/gibbs/pi/krishnaswamy_smita/xingzhi/.conda_envs/pyg/lib/python3.9/site-packages/lightning_fabric/plugins/environments/slurm.py:191: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /gpfs/gibbs/pi/krishnaswamy_smita/xingzhi/.conda_env ...
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name        | Type               | Params
---------------------------------------------------
0 | layers      | ModuleList         | 102   
1 | batch_norm  | BatchNorm          | 88.8 K
2 | fc1         | Linear             | 986 M 
3 | fc2         | Linear             | 444 K 
4 | relu        | ReLU               | 0     
5 | batch_norm1 | BatchNorm1d        | 44.4 K
6 | mlp         | Sequential         | 986 M 
7 | accurac

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

RuntimeError: stack expects each tensor to be equal size, but got [2750, 7403] at entry 0 and [654, 7403] at entry 1

In [ ]:
x, edge_index, edge_attr, y, batch_idx = batch.x, batch.edge_index, batch.edge_attr, batch.y, batch.batch

In [ ]:
node_feat, he_feat = model(x=x, hyperedge_index=edge_index, hyperedge_attr=edge_attr, batch=batch_idx)

RuntimeError: stack expects each tensor to be equal size, but got [2750, 7403] at entry 0 and [581, 7403] at entry 1

In [ ]:
x.size()

torch.Size([1345, 7403])

In [ ]:
edge_index.size()

torch.Size([2, 1301])

In [ ]:
edge_attr.size()

torch.Size([2750, 7403])